🐍 Python OOP — Object Oriented Programming

These are tips to writing better OOP Python applications

## 1. Classes and Instances

**Description:**  
A class defines the blueprint for an object - its attributes and behaviors.  An instance is a concrete object created from that class or an instantiation of the class.

A best practice when creating classes and in OOP would be to use Pydantic for:
- Strong typing
- Automatic runtime validation
- Clear error mesages
- Immutable or mutable models
- Cleaner code than manual validation

** Don't forget to install Pydantic **
From the terminal enter:
uv add pydantic

## Pydantic in this example

Pydantic is a Python library used for validating and parsing data using type annotations. In this notebook, the class `Employee` inherits from `BaseModel`, which tells Pydantic to treat the class as a validated data model.

When we create an instance such as:

```python
emp1 = Employee(first_name="Fred", last_name="Flintstone", gender=Gender.MALE)
```

Pydantic does several important things automatically:

- It checks that each field has the correct type.
  - `first_name` must be a string
  - `last_name` must be a string
  - `gender` must be a valid `Gender` enum value
- It raises a `ValidationError` if the data is not valid.
- It stores the validated values as attributes on the object.
- It gives us a clean, Pythonic way to work with structured data.

The `Field(...)` syntax is used to define metadata for each field. For example:

```python
first_name: str = Field(..., description="First name of the employee")
```

This means the field is required (`...`) and includes a description. The purpose is not only validation, but also clarity for developers and tools.

The custom validator:

```python
@field_validator("first_name", "last_name")
@classmethod
def validate_name(cls, value: str) -> str:
    if not value.strip():
        raise ValueError("Name cannot be empty")
    return value
```

makes sure the names are not blank strings. If someone creates an employee with an empty or whitespace-only name, the model will reject it with a clear error.

The class `Employee` is the blueprint for an employee record. It defines the shape or structure of the data and the rules that the data must follow. The instances (`emp1`, `emp2`) are concrete employee objects created from that blueprint.

Each instance has its own values for the fields and can use the method:

```python
def describe(self) -> str:
    return f"Employee(name={self.last_name}, {self.first_name}, gender={self.gender.value})"
```

This method formats the employee information as a readable string.

### What `from __future__ import annotations` does

The line:

```python
from __future__ import annotations
```

is a Python future import. It tells Python to store annotations as strings instead of evaluating them immediately. This is useful for forward references, especially when types refer to classes that are defined later or when you want to avoid runtime evaluation issues in type hints.

In simple terms, it makes annotations more flexible and prevents problems when Python tries to resolve names before they are fully defined. In this notebook, it helps make the type-hint setup more robust and future-friendly.

### Summary

`Employee` is a validated data model. Pydantic enforces the rules for each field before the object is created. The class defines the structure and validation rules, and each instance represents one employee with real data. The `__future__ import` makes the type annotations safer and easier to work with in modern Python.


In [8]:
from __future__ import annotations
import json
from datetime import date
from enum import Enum
# ***** Don't forget to install pydantic using uv add pydantic if you haven't already *****
from pydantic import BaseModel, Field, field_validator

class Gender(str, Enum):
    """Enumeration for gender options."""
    MALE = "male"
    FEMALE = "female"
    OTHER = "other"

class Employee(BaseModel):
    """Represents an employee with strong typing and automatic validation."""

    first_name: str = Field(..., description="First name of the employee")
    last_name: str = Field(..., description="Last name of the employee")
    gender: Gender = Field(..., description="Gender of the employee")
    date_of_birth: date = Field(..., description="Date of birth of the employee")

    @field_validator("first_name", "last_name")
    @classmethod
    def validate_name(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("Name cannot be empty")
        return value

    @field_validator("date_of_birth")
    @classmethod
    def validate_date_of_birth(cls, value: date) -> date:
        today = date.today()

        if value >= today:
            raise ValueError("Date of birth must be before today")

        if value.year < 1900:
            raise ValueError("Date of birth cannot be before 1900")

        return value

    def _calculate_age(self) -> int:
        today = date.today()
        age = today.year - self.date_of_birth.year

        if (today.month, today.day) < (self.date_of_birth.month, self.date_of_birth.day):
            age -= 1

        return age

    @property
    def age(self) -> int:
        return self._calculate_age()

    def _describe(self) -> str:
        payload = self.model_dump(mode="json")
        payload["age"] = self.age
        return json.dumps(payload)

    def __str__(self) -> str:
        return self._describe()

    def __repr__(self) -> str:
        return self._describe()

#----------- Create Instances -------------------
emp1 = Employee(
    first_name="Fred",
    last_name="Flintstone",
    gender=Gender.MALE,
    date_of_birth=date(1965, 3, 18),
)
emp2 = Employee(
    first_name="Wilma",
    last_name="Flintstone",
    gender=Gender.FEMALE,
    date_of_birth=date(1968, 7, 7),
)
# This will raise a validation error because the DOB is not before today
# emp3 = Employee(first_name="Barney", last_name="Rubble", gender=Gender.MALE, date_of_birth=date.today())

# This will also raise a validation error because the DOB is in the future
# emp4 = Employee(
#     first_name="Pebbles",
#     last_name="Flintstone",
#     gender=Gender.FEMALE,
#     date_of_birth=date.today().replace(year=date.today().year + 1),
# )

print(emp1)
print(emp2)
print(f"{emp1.first_name}'s age is {emp1.age}")


{"first_name": "Fred", "last_name": "Flintstone", "gender": "male", "date_of_birth": "1965-03-18", "age": 61}
{"first_name": "Wilma", "last_name": "Flintstone", "gender": "female", "date_of_birth": "1968-07-07", "age": 58}
Fred's age is 61


## date_of_birth and class validation

The `date_of_birth` field is a required attribute of the `Employee` model. It is declared as:

```python
date_of_birth: date = Field(..., description="Date of birth of the employee")
```

This tells Pydantic that every `Employee` must include a valid `date` value. Because the field type is `date`, Pydantic will enforce that the value is actually a Python `date` object and not a string or some other type. The `Field(...)` syntax marks the field as required, so the object cannot be created without it.

We also add a custom validator:

```python
@field_validator("date_of_birth")
@classmethod
def validate_date_of_birth(cls, value: date) -> date:
    today = date.today()

    if value >= today:
        raise ValueError("Date of birth must be before today")

    if value.year < 1900:
        raise ValueError("Date of birth cannot be before 1900")

    return value
```

This is a class method because it is decorated with `@classmethod`. In Pydantic, this is the pattern used to validate a specific field. The method receives the class (`cls`) and the incoming field value (`value`). It checks the value and raises a `ValueError` if the data is invalid. If it passes validation, it returns the same value so the model continues to build normally.

The private helper method is used internally to calculate the age:

```python
def _calculate_age(self) -> int:
    today = date.today()
    age = today.year - self.date_of_birth.year

    if (today.month, today.day) < (self.date_of_birth.month, self.date_of_birth.day):
        age -= 1

    return age
```

This method is not meant to be called directly by outside code. It is a private helper that keeps the age logic in one place. We access it internally from the public property:

```python
@property
def age(self) -> int:
    return self._calculate_age()
```

The `@property` decorator makes `age` behave like an attribute instead of a method. That means we can read it as `emp1.age` instead of calling something like `emp1.age()`. The property hides the internal implementation details and exposes a clean, readable API.

The same pattern is used for the JSON output. We do not expose a public `describe()` method to external callers. Instead, the class uses internal helpers and the dunder methods:

```python

def _describe(self) -> str:Together, these features show a common OOP pattern: the class defines the rules for valid data, and each instance stores specific values that can be validated and processed.

    payload = self.model_dump(mode="json")

    payload["age"] = self.ageThis is a good OOP pattern because the outside world only sees the public property `age` and the readable object representation, while the class keeps the internal calculation and formatting logic private. The code is cleaner, easier to use, and the implementation can change without affecting callers.

    return json.dumps(payload)

This makes `describe()` effectively internal-only, while `print(emp1)` and the Python REPL use `__str__` and `__repr__` as the public representation hooks. In other words, callers see the JSON output through the normal Python string/representation flow, but the implementation remains encapsulated and the class keeps a cleaner public interface.

    def __str__(self) -> str:

        return self._describe()```

        return self._describe()
    def __repr__(self) -> str: